# Паттерны проектирования

## Цели лекции

После этой лекции вы сможете:
- объяснить, что такое паттерн проектирования и какую проблему он решает;
- различать порождающие, структурные и поведенческие паттерны;
- применять 6 ходовых паттернов: Фабрика, Одиночка, Декоратор, Адаптер, Стратегия, Наблюдатель;
- понимать, где динамизм Python делает паттерн легче – и не переусложнять код;
- выбирать решение по изменению в задаче, проверять порядок обёрток и поведение при ошибках;
- передавать зависимости явно и проверять сервис без внешней отправки.


## Что такое паттерн проектирования

**Паттерн** – типовое, проверенное временем решение повторяющейся проблемы проектирования.
Это не готовый код, а **идея/схема**, которую вы адаптируете под задачу. Паттерны дают общий
словарь: сказать «здесь Наблюдатель» короче, чем объяснять устройство.

Классический каталог из 23 паттернов описали Эрих Гамма, Ричард Хелм,
Ральф Джонсон и Джон Влиссидес. Их называют **Gang of Four**
(GoF, «Банда четырёх»). Паттерны из каталога делят на три группы:
- **порождающие** – как создавать объекты (Фабрика, Одиночка, Строитель…);
- **структурные** – как объекты соединяются (Адаптер, Декоратор, Фасад…);
- **поведенческие** – как объекты взаимодействуют (Стратегия, Наблюдатель, Команда…).

> В Python реализация паттерна часто получается короче: готовые возможности языка
> могут выполнять роль отдельных классов. Это увидим на примерах ниже.

## Порождающие: Фабрика

**Проблема:** нужный класс становится известен только во время работы – из настройки,
ввода пользователя, ответа сервера, – а код, создающий объекты, жёстко завязан на
конкретные классы.

**Решение:** вынести создание в отдельную функцию или метод – «фабрику»,
которая по параметру возвращает нужный объект. Новый класс подключается правкой
фабрики, а не всех мест создания.

Механизм знаком по лекции 7: класс – обычный объект, его можно положить в словарь
и вызвать. Метод `from_string` оттуда же – тоже фабрика, только для одного класса.

In [1]:
class Dog:
    def sound(self): return "гав"

class Cat:
    def sound(self): return "мяу"

def make_animal(kind: str):        # фабрика скрывает выбор класса от вызывающего кода
    return {"dog": Dog, "cat": Cat}[kind]()

for k in ["dog", "cat"]:
    print(k, "->", make_animal(k).sound())

dog -> гав
cat -> мяу


## Порождающие: Одиночка (Singleton)

**Проблема:** нужен ровно один экземпляр на всё приложение (конфиг, пул соединений).

**Решение:** класс сам контролирует своё создание. Классический вариант ниже
использует `__new__`.

В Python отдельный класс часто не нужен: при первом импорте Python создаёт объект
модуля, а следующие импорты в том же процессе возвращают этот же объект. Поэтому
переменные модуля уже хранят общее состояние.

Поле `data` заполняем в `__new__`, при первом создании. В `__init__` его ставить
нельзя: Python вызывает `__init__` при каждом `Config()`, даже когда `__new__`
вернул уже существующий объект, – и общий словарь каждый раз обнулялся бы.

In [2]:
class Config:
    _instance = None
    def __new__(cls):
        if cls._instance is None:          # создаём только при первом вызове
            cls._instance = super().__new__(cls)
            cls._instance.data = {}
        return cls._instance

a = Config(); a.data["debug"] = True
b = Config()                                # тот же самый объект
print("это один объект:", a is b, "| b.data:", b.data)

это один объект: True | b.data: {'debug': True}


### Общие настройки связывают независимые сценарии

Первый сценарий включает `debug`. Второй сам настройку не меняет и ожидает
обычный текст. Но оба обращения видят один `Config`, поэтому второй сценарий
тоже получает префикс.

Проверка зависит от того, какой код выполнился раньше: единственность объекта
сама по себе не обеспечивает изоляцию сценариев.


In [1]:
def format_label(text):
    prefix = "[debug] " if Config().data.get("debug", False) else ""
    return prefix + text


Config().data.clear()
Config().data["debug"] = True
print("Первый сценарий включил debug:", format_label("Готово"))

expected = "Готово"
received = format_label("Готово")
print("Второй сценарий ожидал:", expected)
print("Второй сценарий получил:", received)
print("Ожидание выполнилось:", received == expected)


Первый сценарий включил debug: [debug] Готово
Второй сценарий ожидал: Готово
Второй сценарий получил: [debug] Готово
Ожидание выполнилось: False


### Передаём настройки явно

Передадим конфигурацию аргументом. Место вызова решает,
какие настройки использовать; функция больше не обращается к `Config()`.

Это простой вариант **передачи зависимостей** (dependency injection).
Сценарии могут использовать разные словари. Передача аргумента
не копирует объект: общий словарь по-прежнему оставался бы общим.


In [2]:
def format_label(text, settings):
    prefix = "[debug] " if settings.get("debug", False) else ""
    return prefix + text


debug_settings = {"debug": True}
normal_settings = {"debug": False}
print("Отладка:", format_label("Готово", debug_settings))
print("Обычный режим:", format_label("Готово", normal_settings))
print("Повтор отладки:", format_label("Готово", debug_settings))


Отладка: [debug] Готово
Обычный режим: Готово
Повтор отладки: [debug] Готово


## Структурные: Декоратор

**Проблема:** нужно добавлять поведение объекту, не меняя его класс и не плодя подклассы.

**Решение:** обернуть объект в «декоратор» с тем же интерфейсом, который добавляет своё.

Не путайте два значения слова. Синтаксис `@decorator` из лекции 6 преобразует
функцию при её определении. Паттерн Декоратор соединяет объекты через композицию.
Общая идея – обёртка, но механизм и область применения различаются.

In [3]:
class Coffee:
    def cost(self): return 100
    def desc(self): return "кофе"

class MilkDecorator:               # обёртка с тем же интерфейсом (cost/desc)
    def __init__(self, wrapped): self._w = wrapped
    def cost(self): return self._w.cost() + 30
    def desc(self): return self._w.desc() + " + молоко"

class SugarDecorator:
    def __init__(self, wrapped): self._w = wrapped
    def cost(self): return self._w.cost() + 10
    def desc(self): return self._w.desc() + " + сахар"

drink = SugarDecorator(MilkDecorator(Coffee()))   # обёртки складываются
print(drink.desc(), "=", drink.cost())

кофе + молоко + сахар = 140


### Две операции с ценой: скидка и сбор

У добавок молока и сахара перестановка не меняла итоговую цену.
Теперь одна обёртка уменьшает текущую цену на 10 %, а другая добавляет 20.

Сохраняем интерфейс `cost()`/`desc()` уже написанного `Coffee`.
Каждая обёртка использует результат вложенного объекта.


In [3]:
class Discount:
    def __init__(self, wrapped): self._w = wrapped
    def cost(self): return round(self._w.cost() * 0.9, 2)
    def desc(self): return self._w.desc() + " со скидкой"


class ServiceFee:
    def __init__(self, wrapped): self._w = wrapped
    def cost(self): return self._w.cost() + 20
    def desc(self): return self._w.desc() + " со сбором"


### Обёртки выполняются изнутри наружу

В `Discount(ServiceFee(Coffee()))` скидка применяется и к стоимости сбора:
`(100 + 20) × 0,9 = 108`.

В `ServiceFee(Discount(Coffee()))` сбор добавляется после скидки:
`100 × 0,9 + 20 = 110`.

Порядок обёрток выбирают по правилу расчёта и закрепляют проверкой.


In [4]:
fee_then_discount = Discount(ServiceFee(Coffee()))
discount_then_fee = ServiceFee(Discount(Coffee()))

print("Скидка после сбора:", fee_then_discount.cost())
print("Сбор после скидки:", discount_then_fee.cost())
assert fee_then_discount.cost() == 108
assert discount_then_fee.cost() == 110
print("Исходный кофе:", Coffee().cost())


Скидка после сбора: 108.0
Сбор после скидки: 110.0
Исходный кофе: 100


## Структурные: Адаптер

**Проблема:** есть класс с «неправильным» интерфейсом (сторонняя библиотека, легаси), а код
ждёт другой.

**Решение:** адаптер оборачивает старый объект и предоставляет ожидаемый интерфейс.

In [4]:
class OldPrinter:                  # старый несовместимый интерфейс
    def print_text(self, t): return f"[old] {t}"

class NewPrinter:                  # новый ожидаемый интерфейс: .render()
    def render(self, t): return f"[new] {t}"

class PrinterAdapter:              # адаптер приводит старый интерфейс к новому
    def __init__(self, old): self._old = old
    def render(self, t): return self._old.print_text(t)

def use(printer, text): return printer.render(text)   # код ждёт метод .render()

print(use(NewPrinter(), "привет"))
print(use(PrinterAdapter(OldPrinter()), "привет"))

[new] привет
[old] привет


### Делегирование, Декоратор и Адаптер

Делегирование из лекции 7б – общий механизм: обёртка передаёт работу вложенному объекту.
Декоратор и Адаптер используют этот механизм для разных задач.

| Приём | Интерфейс обёртки | Что меняется |
| --- | --- | --- |
| Декоратор | тот же | добавляется поведение |
| Адаптер | другой, ожидаемый клиентом | согласуются несовместимые интерфейсы |

Обе обёртки содержат исходный объект, то есть используют композицию вместо новой
иерархии наследования.

## Поведенческие: Стратегия

**Проблема:** нужно менять алгоритм на лету (разные способы сортировки, оплаты, сжатия).

**Решение:** вынести алгоритм в отдельный объект или функцию и передавать его.

В лекции 5 мы уже передавали правило функцией в `count_matching(items, rule)`.
Теперь называем этот способ проектирования **Стратегией**. В Python функция уже
подходит на роль стратегии, поэтому отдельные классы обычно избыточны.

In [5]:
def by_price(item): return item["price"]
def by_name(item): return item["name"]

def show(items, key):              # стратегия передаётся как функция
    return [i["name"] for i in sorted(items, key=key)]

data = [{"name": "B", "price": 30}, {"name": "A", "price": 50}, {"name": "C", "price": 10}]
print("по цене:", show(data, by_price))
print("по имени:", show(data, by_name))

по цене: ['C', 'B', 'A']
по имени: ['A', 'B', 'C']


## Поведенческие: Наблюдатель (Observer)

**Проблема:** при изменении одного объекта нужно уведомить много других, не завязывая их жёстко.

**Решение:** объект-издатель хранит список подписчиков и оповещает их о событиях (основа pub/sub,
событийных систем, GUI). Подписчиками служат функции и привязанные методы – колбэки
из лекции 7б.

In [6]:
class Publisher:
    def __init__(self): self._subs = []
    def subscribe(self, fn): self._subs.append(fn)      # подписаться
    def publish(self, event):
        for fn in self._subs: fn(event)                 # оповестить всех

orders = Publisher()
orders.subscribe(lambda e: print("логгер:", e))
orders.subscribe(lambda e: print("уведомление на почту:", e))
orders.publish("заказ #42 создан")

логгер: заказ #42 создан
уведомление на почту: заказ #42 создан


### Ошибка подписчика останавливает уведомление

В существующем `Publisher.publish` обработчики вызываются последовательно.
Если один возбуждает исключение, следующие вызваны не будут.

Это разумно, когда продолжать работу после сбоя нельзя.
Для независимых уведомлений может понадобиться другая политика.


In [5]:
received_events = []

def broken_handler(event):
    raise ValueError("обработчик недоступен")

def remember_event(event):
    received_events.append(event)


stopped = Publisher()
stopped.subscribe(broken_handler)
stopped.subscribe(remember_event)
try:
    stopped.publish("заказ создан")
except ValueError as error:
    print("Сбой:", error)
print("Второй обработчик получил:", received_events)


Сбой: обработчик недоступен
Второй обработчик получил: []


### Продолжаем уведомление и возвращаем ошибки

Если обработчики независимы, можно вызвать остальных и вернуть список ошибок.
Решение о дальнейших действиях остаётся у вызывающего кода.

Исключения не исчезают молча. Такая политика не отменяет действия,
которые обработчики уже успели выполнить.


In [6]:
class CollectingPublisher(Publisher):
    def publish(self, event):
        errors = []
        for fn in list(self._subs):
            try:
                fn(event)
            except Exception as error:
                errors.append(error)
        return errors


continuing = CollectingPublisher()
continuing.subscribe(broken_handler)
continuing.subscribe(remember_event)
received_events.clear()
failures = continuing.publish("заказ создан")
print("Второй обработчик получил:", received_events)
print("Ошибки:", [str(error) for error in failures])


Второй обработчик получил: ['заказ создан']
Ошибки: ['обработчик недоступен']


## Частые ошибки

- **Паттерн ради паттерна.** Цель – решить задачу, а не «применить Singleton». Простой код
  без паттерна часто лучше, чем паттерн без нужды (over-engineering).
- **Буквальный перенос из Java/C++.** В Python Стратегией может быть функция,
  Одиночкой – модуль, а обёрткой – функция или объект. Тяжёлые иерархии обычно не нужны.
- **Смешение Декоратора и `@decorator`.** Паттерн сохраняет интерфейс вложенного объекта;
  синтаксис `@` применяет вызываемый объект при определении функции или класса.
- **Singleton как «глобальная переменная».** Скрытое общее состояние усложняет тесты –
  применяйте осознанно.

### Выбираем решение по изменению в задаче

Для каждой ситуации выберите подходящий приём и назовите,
**какую часть программы он позволит менять отдельно**.

| Ситуация | Что требуется |
| --- | --- |
| Формат выгрузки задан в настройках | Создать CSVWriter или PDFWriter |
| У службы доставки несколько правил расчёта срока | Менять расчёт, сохраняя код оформления заказа |
| Чужая библиотека имеет `draw_text`, наш код ждёт `render` | Подключить библиотеку к существующему коду |
| Нужно измерять время `save()` | Добавить замер, сохранив способ вызова |


### Проверяем выбор по границе изменений

| Ситуация | Выбор и причина |
| --- | --- |
| Создание нужного Writer | Фабрика скрывает выбор конкретного класса |
| Правило расчёта срока | Стратегия заменяет алгоритм внутри прежнего процесса |
| `draw_text` вместо ожидаемого `render` | Адаптер согласует интерфейсы |
| Замер времени `save()` | Декоратор добавляет действие вокруг прежнего вызова |

Приёмы можно сочетать: фабрика способна создать адаптер.
Если отдельной изменяемой части пока нет, дополнительная обёртка может быть лишней.


## Мини-упражнения

В первых трёх задачах примените отдельные приёмы, в четвёртой соедините их
в сервис и проверьте его через подставной отправитель.


### 1. Стратегия: скидки

Напишите `apply_discount(price, strategy)`, где `strategy` – **функция** расчёта цены.
Реализуйте три стратегии: `no_discount` (без скидки), `percent_10` (−10 %),
`fixed_50` (−50, но не ниже 0). Примените их к цене `200`.

In [7]:
# Решение: стратегия – это функция (классы избыточны)
def no_discount(price): return price
def percent_10(price):  return round(price * 0.9, 2)
def fixed_50(price):    return max(0, price - 50)

def apply_discount(price, strategy):   # strategy – функция-стратегия
    return strategy(price)

for s in (no_discount, percent_10, fixed_50):
    print(f"{s.__name__:12} -> {apply_discount(200, s)}")

no_discount  -> 200
percent_10   -> 180.0
fixed_50     -> 150


### 2. Наблюдатель: датчик с отпиской

Сделайте класс `Sensor` с `subscribe(fn)`, `unsubscribe(fn)` и `publish(name, value)`.
Подпишите `alarm` (предупреждение при `value > 40`) и логгер (печатает всегда).
Опубликуйте `42`, отпишите логгер, опубликуйте `45` и `20`.

Отписка – та часть Наблюдателя, которой нет в разборе выше, и именно на ней
в реальных системах текут утечки: подписчик давно не нужен, а издатель
держит на него ссылку.


In [8]:
# Решение: к издателю из разбора добавлена отписка
class Sensor:
    def __init__(self):
        self._subs = []

    def subscribe(self, fn):
        self._subs.append(fn)

    def unsubscribe(self, fn):
        self._subs.remove(fn)           # нет такого подписчика – ValueError

    def publish(self, name, value):
        for fn in list(self._subs):     # копия: подписчик вправе отписаться прямо в обработчике
            fn(name, value)


def alarm(name, value):
    if value > 40:
        print(f"ВНИМАНИЕ: {name} = {value} (порог 40 превышен)")

def log(name, value):
    print(f"лог: {name} = {value}")

sensor = Sensor()
sensor.subscribe(alarm)
sensor.subscribe(log)

sensor.publish("температура", 42)       # сработают оба
sensor.unsubscribe(log)                 # отписка по ссылке на функцию – поэтому логгер не lambda
sensor.publish("температура", 45)       # логгер отписан: только предупреждение
sensor.publish("температура", 20)       # ниже порога: не печатает никто


ВНИМАНИЕ: температура = 42 (порог 40 превышен)
лог: температура = 42
ВНИМАНИЕ: температура = 45 (порог 40 превышен)


### 3. Адаптер: итерируемая обёртка

Есть класс с методом `get_data()`, возвращающим список, а ваш код хочет объект,
по которому можно **итерироваться** в `for`. Напишите адаптер с `__iter__`,
чтобы `for x in Adapter(source)` работал.


In [9]:
# Решение: адаптер отдаёт итератор того списка, что вернул старый объект
class DataSource:                       # чужой класс: отдаёт список методом get_data()
    def get_data(self):
        return [10, 20, 30]

class IterableAdapter:                  # адаптер добавляет ожидаемый интерфейс
    def __init__(self, source):
        self._source = source

    def __iter__(self):
        return iter(self._source.get_data())


for x in IterableAdapter(DataSource()):
    print(x)

print("работает везде, где нужен итерируемый объект:", list(IterableAdapter(DataSource())))


10
20
30
работает везде, где нужен итерируемый объект: [10, 20, 30]


### 4. Сервис уведомлений: соединяем несколько приёмов

Есть два отправителя ниже. Они лишь формируют строки; внешних отправок нет.

1. Напишите `GatewayAdapter` с методом `send(recipient, text)`.
2. Фабрика `make_sender(channel)` выбирает `console` или `legacy`;
   для неизвестного канала возбуждает `ValueError`.
3. `OrderNotifier(sender).notify(order_id, recipient)` передаёт отправителю
   текст `Заказ #42 готов` для номера 42.
4. Проверьте адресата, текст и число вызовов через отправитель со списком сообщений.


In [7]:
class LegacyGateway:
    def deliver(self, text, to):
        return f"legacy -> {to}: {text}"


class ConsoleSender:
    def send(self, recipient, text):
        return f"console -> {recipient}: {text}"


### Решение: адаптер и выбор отправителя

Адаптер согласует имя метода и порядок аргументов.
Фабрика отвечает только за создание нужного отправителя.
Оба результата фабрики поддерживают `send(recipient, text)`.


In [8]:
class GatewayAdapter:
    def __init__(self, gateway):
        self.gateway = gateway

    def send(self, recipient, text):
        return self.gateway.deliver(text, to=recipient)


def make_sender(channel):
    if channel == "console":
        return ConsoleSender()
    if channel == "legacy":
        return GatewayAdapter(LegacyGateway())
    raise ValueError(f"Неизвестный канал: {channel}")


### Решение: сервис получает готовую зависимость

`OrderNotifier` формирует сообщение, но не выбирает канал отправки.
Создание зависимости находится в месте сборки приложения.

Чтобы сменить канал, достаточно передать другой отправитель;
код формирования сообщения остаётся прежним.


In [9]:
class OrderNotifier:
    def __init__(self, sender):
        self.sender = sender

    def notify(self, order_id, recipient):
        text = f"Заказ #{order_id} готов"
        return self.sender.send(recipient, text)


notifier = OrderNotifier(make_sender("legacy"))
print(notifier.notify(42, "student"))


legacy -> student: Заказ #42 готов


### Проверка: отправитель записывает обращения

В проверке передадим объект с тем же методом `send`, который сохраняет аргументы.
Фабрика для этого не нужна: зависимость можно передать напрямую.

Проверяем наблюдаемое действие сервиса: кому, какой текст и сколько раз передан.


In [10]:
class RecordingSender:
    def __init__(self):
        self.messages = []

    def send(self, recipient, text):
        self.messages.append((recipient, text))


recording = RecordingSender()
tested = OrderNotifier(recording)
tested.notify(42, "student")
assert recording.messages == [("student", "Заказ #42 готов")]
tested.notify(43, "teacher")
assert recording.messages == [
    ("student", "Заказ #42 готов"),
    ("teacher", "Заказ #43 готов"),
]
print("Проверены два уведомления:", len(recording.messages))


Проверены два уведомления: 2


## Что почитать

- [Refactoring.Guru](https://refactoring.guru/ru/design-patterns) – каталог паттернов
  на русском, с примерами и разбором «когда не надо».
- [Python Design Patterns](https://python-patterns.guide/) – Брэндон Роудс о том,
  какие паттерны в Python вырождаются в одну строку.
- Э. Гамма и др., «Приёмы объектно-ориентированного проектирования» – первоисточник (GoF).
